# NVIDIA Dynamo Qwen3 8B Model

This notebook shows how to use the [Dynamo](../../../../../charts/machine-learning/serving/dynamo/) Helm chart to serve [Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B) model with [NVIDIA Dynamo](https://github.com/ai-dynamo/dynamo), first **aggregated** on a single GPU, then **disaggregated** with prefill and decode as separate workers.

Unlike the Ray Serve and Triton examples, there is no container to build: the runtime image is pulled anonymously from `nvcr.io`. Qwen3-8B is also not a gated model, so no Hugging Face token is required.

## Setup and Imports

In [ ]:
! pip install kubernetes
! pip install openai

In [ ]:
import os
import subprocess
import sys

# Set working directory
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow'))
print(f"Working directory: {os.getcwd()}")

# Get the src directory
src_dir = os.path.join(os.getcwd(), "src")
sys.path.insert(0, src_dir)

from k8s.utils import (
    wait_for_helm_release_pods,
    wait_for_dynamo_ready,
    find_k8s_service
)

# Get notebook directory
notebook_dir = os.path.join(os.getcwd(), 'examples', 'inference', 'dynamo', 'vllm', 'qwen3-8b')
print(f"Notebook directory: {notebook_dir}")

# initialize key variables
release_name = 'dyn-qwen3-8b'
namespace = 'kubeflow-user-example-com'
hf_model_id = 'Qwen/Qwen3-8B'
served_model_name = 'qwen3-8b'

## Step 1: Verify the Dynamo Platform

The Dynamo operator is installed by Terraform from the local [dynamo-platform](../../../../../charts/dynamo-platform/) chart, so there is nothing to install here. This cell confirms it is running before we depend on it.

The operator is **off by default**. If the controller pod or the CRDs are missing, set `dynamo_enabled = true` in your `.tfvars` and re-apply the Terraform configuration.

In [ ]:
# Confirm the operator is running and its CRDs are registered
cmd = ['kubectl', 'get', 'pods', '-n', 'dynamo-system']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)

cmd = ['kubectl', 'get', 'crd', '-o', 'name']
result = subprocess.run(cmd, capture_output=True, text=True)
crds = [line for line in result.stdout.splitlines() if 'nvidia.com' in line]
print(f"Dynamo CRDs registered: {len(crds)}")
for crd in crds:
    print(f"  {crd}")

assert crds, "No nvidia.com CRDs found. Is the dynamo-platform Helm release installed?" 

## Step 2: Download Hugging Face Qwen3 8B Pre-trained Model Weights

Qwen3-8B is not a gated model, so unlike the other examples in this repository no Hugging Face token is needed. The weights land at `/fsx/pretrained-models/Qwen/Qwen3-8B`, which is the path each values file reads.

In [ ]:
cmd = [
    'helm', 'install', '--debug', f'{release_name}-weights',
    'charts/machine-learning/model-prep/hf-snapshot',
    '--set-json', f'env=[{{"name":"HF_MODEL_ID","value":"{hf_model_id}"}}]',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# wait for model download job to complete
wait_for_helm_release_pods(release_name=f'{release_name}-weights', namespace=namespace)

In [ ]:
# Uninstall the Helm chart after completion
cmd = ['helm', 'uninstall', f'{release_name}-weights', '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

## Step 3: Launch Aggregated Serving

Prefill and decode run in the same process on one GPU. This is the simplest configuration and the one to get working first.

Karpenter provisions one single-GPU node — the cheapest instance type in the values file's `node_types` list that has capacity, so which type you get varies between runs. Allow ~5 minutes for the node, the image pull and the vLLM engine to load.

In [ ]:
cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-agg.yaml',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Wait for the DynamoGraphDeployment to be ready
wait_for_dynamo_ready(release_name=release_name, namespace=namespace)

## Step 4: Check Service Status

In [ ]:
# Check service status
service_name = f"{release_name}-frontend"
service = find_k8s_service(service_name, namespace)
if service:
    print(f"Service {service.metadata.name} is available.")
    print(f"Service type: {service.spec.type}")
    print(f"Service ports: {service.spec.ports} ")
    print(f"Run\n\n kubectl port-forward -n {namespace} svc/{service.metadata.name}  8000:8000 \n\nin a new terminal")
else:
    print(f"Service {service_name} not found.")

## Test Service with OpenAI API

Test the service using OpenAI API for completion. This test assumes you are doing a `kubectl port-forward` to the service on `localhost` port `8000`.

First confirm the worker has registered its model with the frontend. An empty list here means the frontend is up but no worker has registered yet — check the worker pod's logs rather than the frontend's.

In [ ]:
import time
from openai import OpenAI, APIConnectionError

client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

def list_models(client, attempts=30, interval=10):
    """List served models, waiting for the port-forward rather than failing on a refused
    connection. A `kubectl port-forward` is bound to one pod, so it dies whenever the frontend
    pod is replaced -- which makes a dead tunnel look exactly like a broken deployment."""
    for attempt in range(attempts):
        try:
            return [model.id for model in client.models.list()]
        except APIConnectionError:
            print(f"No answer on localhost:8000 (attempt {attempt + 1}/{attempts}). "
                  f"Is `kubectl port-forward` running against the current frontend pod?")
            time.sleep(interval)
    raise RuntimeError("Endpoint never answered. Start the kubectl port-forward and re-run.")

print(list_models(client))

In [ ]:
response = client.chat.completions.create(
    model=served_model_name,
    messages=[{"role": "user", "content": "Explain the concept of attention masking in causal language models and its importance for autoregressive generation."}],
    max_tokens=2048
)

print(response.choices[0].message.content)

Qwen3 is a reasoning model and emits a `<think>` block before its answer, so keep `max_tokens` generous. At 128 the response is truncated mid-reasoning and looks like a broken deployment rather than a budget you set too low.

## Step 5: Switch to Disaggregated Serving

Prefill and decode run as separate workers exchanging the KV cache over NIXL. This is what Dynamo exists to do: the two phases have very different compute profiles, and separating them stops long prompts from stalling in-flight decodes.

The aggregated deployment must be uninstalled first. This is not optional: the admission webhook rejects any change to the *set of component names* on an existing `DynamoGraphDeployment` with `component topology is immutable and cannot be modified after creation`, so `helm upgrade` from `dgd-agg.yaml` to a disaggregated values file fails.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

Now install the disaggregated topology. [`dgd-disagg-tp1.yaml`](dgd-disagg-tp1.yaml) runs each worker at `TP=1` on its own single-GPU node, so the KV cache crosses the network between pods — 2 GPUs in total.

There is also [`dgd-disagg.yaml`](dgd-disagg.yaml), which runs each worker at `TP=2` on one 4-GPU node. Worth knowing before assuming that makes it faster: on a `g5.12xlarge` the intra-node transfer measured the same order of magnitude as the cross-AZ network rate, because `g5`'s A10G GPUs have no NVLink and peer transfers fall back to PCIe. See [bench/README.md](bench/README.md).

In [ ]:
cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/serving/dynamo/',
    '-f', f'{notebook_dir}/dgd-disagg-tp1.yaml',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Wait for the DynamoGraphDeployment to be ready. This provisions two GPU nodes, so it takes
# longer than the aggregated deployment.
wait_for_dynamo_ready(release_name=release_name, namespace=namespace, timeout=2400)

## Test the Disaggregated Service

The API surface is unchanged, which is the point — disaggregation is a serving-topology decision, not a client-visible one.

**Restart your `kubectl port-forward` before running the next cell.** The reinstall replaced the frontend pod, and a port-forward is bound to the pod it started against, so the old tunnel is dead. The cell below waits for the endpoint instead of failing immediately, but it cannot restart the tunnel for you:

    kubectl port-forward -n kubeflow-user-example-com svc/dyn-qwen3-8b-frontend 8000:8000

In [ ]:
client = OpenAI(base_url="http://localhost:8000/v1", api_key="dummy")

print(list_models(client))

In [ ]:
response = client.chat.completions.create(
    model=served_model_name,
    messages=[{"role": "user", "content": "Explain the concept of attention masking in causal language models and its importance for autoregressive generation."}],
    max_tokens=2048
)

print(response.choices[0].message.content)

## Confirm the Request Really Was Disaggregated

"The pods are Running" is not evidence that disaggregated inference is happening. If the decode worker reports KV transfers but the prefill worker never receives a request, the two are not paired and you are effectively running aggregated on two GPUs.

A served request produces `NIXL compatibility check passed`, a `TransferTopology(...)` line, and `KV Transfer metrics: Num successful transfers=...` with a non-zero throughput in the decode worker's log. The matching `component="prefill"` request appears in the prefill worker's log.

To go further and watch the two phases on the GPUs themselves — which GPU each worker got, whether prefill is compute-bound while decode is bandwidth-bound, and the frontend's own time-to-first-token and inter-token-latency histograms — see [Observing Prefill and Decode](README.md#observing-prefill-and-decode).

In [ ]:
import re

for component_type in ['decode', 'prefill']:
    cmd = ['kubectl', 'logs', '-n', namespace,
           '-l', f'nvidia.com/dynamo-component-type={component_type}', '--tail=400']
    result = subprocess.run(cmd, capture_output=True, text=True)
    matches = [line for line in result.stdout.splitlines()
               if any(k in line for k in ('NIXL', 'TransferTopology', 'KV Transfer metrics'))]
    print(f"===== {component_type} worker: {len(matches)} matching lines =====")
    for line in matches[-6:]:
        print(line)

    # The metrics line is emitted periodically whether or not anything transferred, so the count
    # is what matters -- `Num successful transfers=0` would otherwise read as a pass.
    transfers = max((int(n) for n in re.findall(r'Num successful transfers=(\d+)', result.stdout)),
                    default=0)
    print(f"-> peak successful KV transfers reported: {transfers}")
    if component_type == 'decode':
        assert transfers > 0, ("Decode worker reported no KV transfers. The deployment is "
                               "running but no request was disaggregated -- did the completion "
                               "above actually succeed?")
    print()

## Benchmarking

Whether disaggregation is *faster* than aggregation is a separate question from whether it works, and it depends on your model size and interconnect. At a fixed GPU budget on identical GPUs, aggregated with two replicas won throughput and time-to-first-token, while disaggregation held p99 inter-token latency dramatically lower under prefill-heavy load.

[bench/README.md](bench/README.md) has the method, the findings and the load generator, including the dead ends that shaped the values files.

## Step 6: Stop Service

When you're done with the service, run this cell to clean up resources. Karpenter deprovisions the GPU nodes once the pods are gone, after the `karpenter_consolidate_after` delay.

In [ ]:
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)